Environment

In [ ]:
libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet",
    "pycomet": "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}

In [ ]:
import sys
for lib in libs.values():
    sys.path.append(lib)

Import libraries

In [ ]:
import os
import json
import pycomet
import numpy as np
import tensorflow as tf

from scipy.linalg import expm
from silx.io.dictdump import h5todict

from hypernet import const
from hypernet.apps import utils as hyputils
from pycomet.networks import NodePartitionFunctions, EdgeBackward, LinearMEGraphNet

Plotting

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

path_to_figs = "./figs/"
os.makedirs(path_to_figs, exist_ok=True)

def plot_var(
    x,
    y,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    xlim = [np.amin(x), np.amax(x)]
    ax.set_xlim(xlim)
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=y.shape[0]//20,
        ls='--',
        lw=1
    )
    for d in range(y.shape[1]):
        ax.plot(x, y[:,d], c=COLORS[d], **pred_style)
        ax.plot(xref, yref[:,d], c=COLORS[d], label=names[d], **true_style)
    plt.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.show()
    plt.close()

Initializing thermophysical models

In [ ]:
with open("./../physics/inputs.json") as file:
    phyinp = json.load(file)
# Species thermo
sp_th = hyputils._get_specie_thermos(phyinp)
# Mixture
mix = hyputils._get_mixture(phyinp, sp_th)
# Kinetics
kin = hyputils._get_kinetics(phyinp, sp_th, mix)

Numerical solution

In [ ]:
sol = h5todict(h5file='./../physics/solution.hdf5')

In [ ]:
na, nm = [sol[f"X_{s}"] * sol['n'] for s in sp_th.keys()]

In [ ]:
# Compute source term
t, T = [sol[k] for k in ('t', 'T')]
n = np.hstack([na, nm])
omega = kin.function(t=t, y=[n, T])
dnmdt = omega[:,1:-1] / const.UNA
dnmdt = dnmdt.numpy().squeeze()

Construct graph

In [ ]:
# Adjacency matrix (forward)
kf = 0.0
for reaction in kin.reactions:
    kf = kf + reaction.fwd.k.numpy().squeeze()[0]
k_idx = np.where(kf > 0)
# Dimension
nb_pts, nb_nodes = nm.shape

In [ ]:
# Nodes
node_nm = nm[0].reshape(1,nb_nodes,1)
node_nm = np.tile(node_nm, (nb_pts,1,1))
node_qm = sp_th["O2"].pf.int.Q.numpy()
node_qm = np.expand_dims(node_qm, axis=-1)

In [ ]:
# Edges
edge_kf = kf[k_idx].reshape(1,-1,1)
edge_kf = np.tile(edge_kf, (nb_pts,1,1))
edge_na = na[0].reshape(1,1,1)
edge_na = np.tile(edge_na, (nb_pts,1,1))

In [ ]:
# Graph
graph = [node_nm, node_qm, edge_kf, edge_na]
graph = tf.nest.map_structure(tf.convert_to_tensor, graph)
# Target
target = nm[...,None]

Load linear master equation graph model

In [ ]:
lays_args = {

    "edge_bwd": {
        "lay_id": "EdgeBackward",
        "kwargs": {
            "node_index_pf": [1],
            "static_fwd_edge_index": "./data/fwd_edge_index.txt"
        }
    },

    "lin_me_gnet": {
        "lay_id": "LinearMEGraphNet",
        "kwargs": {
            "node_attr_indices": [0]
        }
    }

}

In [ ]:
tf.keras.backend.clear_session()

lays = {
    "edge_bwd": EdgeBackward(**lays_args["edge_bwd"]["kwargs"]),
    "lin_me_gnet": LinearMEGraphNet(**lays_args["lin_me_gnet"]["kwargs"])
}

def get_graph(inputs):
    nm, qm, kf, na = inputs
    node_attr = tf.concat([nm,qm], axis=-1)
    edge_attr = kf*na
    return (node_attr, edge_attr)

def call(inputs, training=False):
    graph = get_graph(inputs)
    # Edge backward
    graph = lays["edge_bwd"](graph, training=training)
    # Graph processor
    return lays["lin_me_gnet"](graph, training=training) + inputs[0]

In [ ]:
shapes = tf.nest.map_structure(lambda x: tuple(x.shape[1:]), graph)
inputs = [tf.keras.Input(shape) for shape in shapes]
model = tf.keras.Model(inputs, call(inputs))

In [ ]:
def add_noise(x, stddev=0.02):
    rnd = tf.random.get_global_generator()
    dx = rnd.normal(
        shape=tf.shape(x),
        mean=0.0,
        stddev=stddev,
        dtype=x.dtype,
    )
    return x * (1 + dx)

In [ ]:
graphn = tf.nest.map_structure(tf.identity, graph)
graphn[0] = add_noise(graphn[0], stddev=0.2)
with tf.GradientTape(persistent=True) as g:
  g.watch(graphn[2])
  y = model(graphn)
  # e = tf.keras.losses.mean_squared_error(target / 1e23, y / 1e23)
  e = tf.reduce_mean((target / 1e23)**2 - (y / 1e23)**2, axis=0)
# dy_dx = g.batch_jacobian(e, graphn[1])
dy_dx = g.gradient(e, graphn[2])
dy_dx = tf.squeeze(dy_dx)
dy_dx